## Prepocessing
In this notebook, we prepare the annotated drone tiles for use in finetuning the deepforest model. The tiles are provided in TIF format while the annotations are standard COCO JSON files, one for each tile. In order to use these for training, we will need to extract the annotations and put them into two CSV files - one for training and another for validation.  

The dataset preparation involves the following steps:
1. Convert the JSON annotation files into equivalent CSV files compatible with DeepForest.
2. Merge the CSV files into a single CSV file containing all the annotations. This merged file .
3. Split these annotations in the single CSV into train/validation portions, split by image rather than rows.

In [33]:
import os
import sys

sys.path.append(os.path.abspath(".."))

import warnings
warnings.filterwarnings('ignore')

import random
import traceback
import numpy as np
import pandas as pd

from glob import glob
from tqdm import tqdm
from PIL import Image

import geopandas as gpd
from shapely.geometry import box

from deepforest import utilities, preprocess
from scripts.coco_to_deepforest import read_coco_to_deepforest

### Step 1 - Convert JSON to CSV

In [51]:
df = read_coco_to_deepforest(
    json_path=r'D:\dsail\lacuna_field_work\Aerial\Phase_1\Annotations\2024_08_1.json', 
    output_csv_path='2024_08_1.csv', 
    root_dir='.',
    include_geometry=False
)
df.head()

,image_path,xmin,ymin,xmax,ymax,label
0,2024_08_1.png,1364,1587,1480,1707,Tree
1,2024_08_1.png,1182,2036,1270,2120,Tree
2,2024_08_1.png,1352,2246,1441,2345,Tree
3,2024_08_1.png,1645,1549,1738,1641,Tree
4,2024_08_1.png,2844,1700,2961,1816,Tree


In [52]:
# loopthrough all JSON files and save them as CSVs
phase = "Phase2"
pattern = f"C:\\Aerial\\{phase}\\Annotations\\*.json"
out_dir = f'csv_annotations\\{phase}'
os.makedirs(out_dir, exist_ok=True)

for path in tqdm(sorted(glob(pattern))):
    out_path = os.path.join(out_dir, os.path.basename(path).split('.')[0] + '.csv')
    read_coco_to_deepforest(path, out_path, include_geometry=False)

100%|██████████| 248/248 [00:00<00:00, 351.82it/s]


### Step 2 - Merge the CSVs

In [53]:
# merge all csv files into one
pattn = f"csv_annotations\\{phase}\\*.csv"
paths = sorted(glob(pattn))

dfs = []
for path in tqdm(paths):
    df = pd.read_csv(path)
    dfs.append(df)

merged_df = pd.concat(dfs, axis=0)
merged_df['image_path'] = merged_df['image_path'].str.replace('png', 'tif')

merged_df.to_csv('2025_merged.csv', index=False)

100%|██████████| 248/248 [00:03<00:00, 75.82it/s]


In [54]:
utilities.read_file('2025_merged.csv', r"C:\Aerial\Phase_1\Tiles")

,image_path,xmin,ymin,xmax,ymax,label,geometry
0,2025_02_01.tif,227,105,310,205,Tree,"POLYGON ((310 105, 310 205, 227 205, 227 105, ..."
1,2025_02_01.tif,329,182,374,228,Tree,"POLYGON ((374 182, 374 228, 329 228, 329 182, ..."
2,2025_02_01.tif,146,301,208,359,Tree,"POLYGON ((208 301, 208 359, 146 359, 146 301, ..."
3,2025_02_01.tif,0,351,70,472,Tree,"POLYGON ((70 351, 70 472, 0 472, 0 351, 70 351))"
4,2025_02_01.tif,833,369,957,488,Tree,"POLYGON ((957 369, 957 488, 833 488, 833 369, ..."
...,...,...,...,...,...,...,...
34654,2025_02_99.tif,3159,3327,3274,3436,Tree,"POLYGON ((3274 3327, 3274 3436, 3159 3436, 315..."
34655,2025_02_99.tif,2935,3922,3006,3987,Tree,"POLYGON ((3006 3922, 3006 3987, 2935 3987, 293..."
34656,2025_02_99.tif,3935,4174,4007,4249,Tree,"POLYGON ((4007 4174, 4007 4249, 3935 4249, 393..."
34657,2025_02_99.tif,2364,4143,2495,4264,Tree,"POLYGON ((2495 4143, 2495 4264, 2364 4264, 236..."


### Step 3 - Split Annotations into Train/Validation/Test Sets
In this case, since the directory containing the images has tiles from only one orthophoto and the test images must be from a different orthophoto, we use a train/val split of 90/10. Afterwards, we use a test set equal in number to the validation set. 

#### 3.1 - Train & Validation Sets

In [6]:
# TODO: set these paths
ALL_ANNOTS_CSV = r"2024_merged.csv"   # must have: image_path,xmin,ymin,xmax,ymax,label
IMAGE_ROOT_DIR = r"D:\dsail\lacuna_field_work\Aerial\Phase_1\Tiles" # folder that contains the images referenced by image_path

OUT_DIR = r"training_dataset"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_CSV = os.path.join(OUT_DIR, "train.csv")
VAL_CSV   = os.path.join(OUT_DIR, "val.csv")

# Load and basic validation
df = pd.read_csv(ALL_ANNOTS_CSV)
required_cols = {"image_path", "xmin", "ymin", "xmax", "ymax", "label"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in {ALL_ANNOTS_CSV}: {sorted(missing)}")

# Important: avoid leakage by splitting at the image level
images = sorted(df["image_path"].dropna().unique().tolist())
if len(images) < 3:
    raise ValueError(f"Need at least 3 unique images to do a train/val split. Found {len(images)}")

random.seed(42)
random.shuffle(images)

val_frac = 0.1
n_val = max(1, int(round(len(images) * val_frac)))
val_images = set(images[:n_val])
train_images = set(images[n_val:])

train_df = df[df["image_path"].isin(train_images)].copy()
val_df   = df[df["image_path"].isin(val_images)].copy()

# Save
train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)

print(f"Unique images: total={len(images)} train={len(train_images)} val={len(val_images)}")
print(f"Rows (boxes): train={len(train_df)} val={len(val_df)}")
print("Wrote:", TRAIN_CSV)
print("Wrote:", VAL_CSV)

Unique images: total=244 train=220 val=24
Rows (boxes): train=23918 val=3288
Wrote: training_dataset\train.csv
Wrote: training_dataset\val.csv


#### 3.2 - Test Set

In [56]:
# TODO: set these paths
ALL_ANNOTS_CSV = r"2025_merged.csv"   # must have: image_path,xmin,ymin,xmax,ymax,label
IMAGE_ROOT_DIR = r"C:\Aerial\Phase2\Tiles" # folder that contains the images referenced by image_path

OUT_DIR = r"training_dataset"
os.makedirs(OUT_DIR, exist_ok=True)

TEST_CSV = os.path.join(OUT_DIR, "test.csv")

# Load and basic validation
df = pd.read_csv(ALL_ANNOTS_CSV)
required_cols = {"image_path", "xmin", "ymin", "xmax", "ymax", "label"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns in {ALL_ANNOTS_CSV}: {sorted(missing)}")

# Important: avoid leakage by splitting at the image level
images = sorted(df["image_path"].dropna().unique().tolist())
if len(images) < 3:
    raise ValueError(f"Need at least 3 unique images to do a train/val split. Found {len(images)}")

random.seed(42)
random.shuffle(images)

n_val = 30
test_images = set(images[:n_val])
test_df = df[df["image_path"].isin(test_images)].copy()

# Save
test_df.to_csv(TEST_CSV, index=False)

print(f"Unique images: total={len(images)} test={len(test_images)}")
print(f"Rows (boxes): train={len(test_df)}")
print("Wrote:", TEST_CSV)

Unique images: total=248 test=30
Rows (boxes): train=4484
Wrote: training_dataset\test.csv


### Step 4 - Splitting the Large Tiles into Small Chips
Now that we have our train/validation/test CSV files, we need to split the large $4325 \times 4325$ tiles into smaller $400 \times 400$ chips which we will use for finetuning the pretrained deepforest model.

In [ ]:
# ----------------------------------------- #
# 1. Setup
# ----------------------------------------- #

LARGE_RASTER_DIR = r"C:\Aerial\Phase1\Tiles"
LARGE_RASTER_CSV = r"training_dataset\train.csv"
CHIPS_DIR = r"training_dataset\train_chips"
OUT_CSV = "master_train.csv"

# LARGE_RASTER_DIR = r"C:\Aerial\Phase1\Tiles"
# LARGE_RASTER_CSV = r"training_dataset\val.csv"
# CHIPS_DIR = r"training_dataset\val_chips"
# OUT_CSV = "master_val.csv"

# LARGE_RASTER_DIR = r"C:\Aerial\Phase2\Tiles"
# LARGE_RASTER_CSV = r"training_dataset\test.csv"
# CHIPS_DIR = r"training_dataset\test_chips"
# OUT_CSV = "master_test.csv"

def split_raster_files(large_raster_path:str, large_raster_csv:str, save_dir:str):
    """
    Split large raster files (TIF) into smaller tiles to be used for training or evaluation and
    save these tiles in a separate directory. The box coorindates in the annotation file are also
    adjusted and saved.
    """
    # create output folder
    os.makedirs(save_dir, exist_ok=True)

    annotations = pd.read_csv(large_raster_csv)
    annotations["geometry"] = annotations.apply(
        lambda row: box(
            row["xmin"],
            row["ymin"],
            row["xmax"],
            row["ymax"],
        ),
        axis=1,
    )
    annotations = gpd.GeoDataFrame(annotations, geometry="geometry")

    # Normalize CSV image names and keep only rasters listed in the CSV
    csv_image_names = set(
        annotations["image_path"]
        .map(os.path.basename)
    )

    # ----------------------------------------- #
    # Filter only those images in the annotations
    # ----------------------------------------- #
    tif_files = [
        path for path in glob(os.path.join(large_raster_path, "*.tif"))
        if os.path.basename(path) in csv_image_names
    ]
    print(f"Found {len(tif_files)} large images to process.")

    all_crop_dfs = []
    skipped_files = []

    # ----------------------------------------- #
    # Loop and Split
    # ----------------------------------------- #
    for tif_path in tqdm(tif_files):
        try:
            image = np.array(Image.open(tif_path).convert("RGB"))
            image = np.moveaxis(image, -1, 0)  # HWC -> CHW

            image_name = os.path.basename(tif_path)
            image_annotations = annotations[annotations["image_path"].map(os.path.basename)==image_name].copy()

            crop_df = preprocess.split_raster(
                numpy_image=image,
                image_name=image_name,
                annotations_file=image_annotations,
                patch_size=400,
                patch_overlap=0,  # 0 overlap for strict evaluation
                save_dir=save_dir
            )
            # print(crop_df)
            
            # If the raster had no trees, split_raster might return None or empty depending on version
            if crop_df is not None and not crop_df.empty:
                all_crop_dfs.append(crop_df)
                
        except Exception as e:
            skipped_files.append((os.path.basename(tif_path), str(e)))

    # ---------------------------------------------------------
    # Merge and Save Master CSV
    # ---------------------------------------------------------
    print("Merging separate dataframes ...")
    if all_crop_dfs:
        master_test_df = pd.concat(all_crop_dfs, ignore_index=True)
        
        # Save this master CSV. This is what you will feed to model.evaluate()
        master_test_df.to_csv(OUT_CSV, index=False)
        
        print(f"\nSuccess! Created {len(master_test_df)} annotated crops.")
        print(f"Master CSV saved to: {OUT_CSV}")
    
    elif skipped_files:
        print(f"\nSkipped files: {len(skipped_files)}")
        # for filename, error in skipped_files:
        #     print(f"{filename}: {error}")

    else:
        print("\nError: No crops were generated. Check your paths and CSV format.")


split_raster_files(LARGE_RASTER_DIR, LARGE_RASTER_CSV, CHIPS_DIR)

Found 220 large images to process.


100%|██████████| 220/220 [07:53<00:00,  2.15s/it]


Merging separate dataframes ...

Success! Created 24026 annotated crops.
Master CSV saved to: master_train.csv
